In [0]:
dbutils.widgets.text("config_file_path", "")

config_file_path = dbutils.widgets.get("config_file_path")

In [0]:
from pyspark.sql.functions import col ,regexp_replace, year , current_date , concat_ws , when , count ,avg, sum , desc , dense_rank
from pyspark.sql.window import Window


In [0]:
%run ./notebook1

Config loaded successfully
Input Path: /Volumes/csvfiles/default/demovol
Bronze Table: csvfiles.default.bronze_table


In [0]:
# Read Silver
df = spark.table(silver_table)

df.display()

Employee_ID,First_Name,Last_Name,Email,Department,Salary,Hire_Date,SSN,SSN_encrypted,Salary_encrypted,ingestion_date,hire_year,processed_date,full_name,salary_band
EMP001,John,Doe,john.doe@example.com,Engineering,85000.0,2022-01-15,123-45-6789,01a54629efb952287e554eb23ef69c52097a75aecc0e3a93ca0855ab6d7a31a0,c08e7a2f63f203781ccfc863b16731b32563ae9b3186cd6062c518e64addf523,2026-05-01,2022,2026-05-01,John Doe,High
EMP006,Sarah,Brown,sarah.b@example.com,Finance,78000.0,2022-09-05,678-90-1234,ba4ae52efcaa4eccd489109cb6a7c231a41f045c504512d26fb169058746c68c,a2a239b2e15e1b283bda38ca3d4fd554c76fb10227bdbf203298a31d28772e42,2026-05-01,2022,2026-05-01,Sarah Brown,Medium
EMP010,Ashley,Jackson,ashley.j@example.com,HR,62000.0,2022-04-14,012-34-5678,af7d09747c8cbe325850a962be1af48ae6ddf23c98476caaa137a29be46562f0,20dd120a7502f16300940eda4f4d4fbdcf860ae54f214921e5133551d61188d5,2026-05-01,2022,2026-05-01,Ashley Jackson,Low
EMP014,Amanda,Thompson,amanda.t@example.com,Marketing,73000.0,2022-10-08,444-55-6666,74e4145b168aa2249718acd393752a0e4e625acc7adb46875d80f82e717ce94e,70125bf7d6cc8f4830742ac6eccb8a0e6c1873737c4fd143a8a28cfe230b7abf,2026-05-01,2022,2026-05-01,Amanda Thompson,Medium
EMP003,Michael,Johnson,michael.j@example.com,Sales,68000.0,2023-03-10,345-67-8901,0159e3ba838b89a0a4bdb66e76c1d269f40e77ceec617d85a14156ac04b7c090,deec7508debf45a898aabd88b15304126f423bc09d22eef998520633faec0d3d,2026-05-01,2023,2026-05-01,Michael Johnson,Low
EMP008,Jessica,Anderson,jessica.a@example.com,Marketing,75000.0,2023-01-12,890-12-3456,5144e522dddd37d151df5328b9cb42c6addfab30bc1e42501c6e153bbfb43ad0,32abe77179b0106aa718c92b3ae0d64f276ca768732735495276c74fa37b8957,2026-05-01,2023,2026-05-01,Jessica Anderson,Medium
EMP013,Charles,Martin,charles.m@example.com,Sales,69000.0,2023-05-25,333-44-5555,5158e06a24728730f7eb7888307ff43585c650a7fdb05e9f5a8f5b750bffbb95,9b13e431bb2aa8f6d770c6d0e607bcb796e9ec57887b98f4aebf803b64d8768e,2026-05-01,2023,2026-05-01,Charles Martin,Low
EMP002,Jane,Smith,jane.smith@example.com,Marketing,72000.0,2021-11-01,234-56-7890,2a87a12699cef0854f5c726f45ced690ba58fc9eef3e72610725a9b26479de42,841c756dd0d6ad0204492278045827b2bd71918d0d798dc27acfffe737385f51,2026-05-01,2021,2026-05-01,Jane Smith,Medium
EMP007,James,Taylor,james.t@example.com,Engineering,88000.0,2021-02-28,789-01-2345,1ecdb8649cfa0aa0b2607ebcfb282360312884153030d41b0605a20b55182845,19d47b70a2e540349d353062a310df4163daff2971edf9d0cc0b16eb5d3ea87c,2026-05-01,2021,2026-05-01,James Taylor,High
EMP012,Megan,Harris,megan.h@example.com,Finance,81000.0,2021-06-17,222-33-4444,3371a2c893de6cad595c43039412ea9d602c6e265583ba5bb9c931047ef3fb40,541574b6248c6085a282d472f720898f3c8bf3604a0b9ef1cbaa4a1a1c9807f4,2026-05-01,2021,2026-05-01,Megan Harris,High


In [0]:
# Aggregation Table

agg_df = df.groupBy("Department").agg(
    count("*").alias("employee_count"),
    sum("Salary").alias("total_salary"),
    avg("Salary").alias("avg_salary")
)

agg_df.display()

Department,employee_count,total_salary,avg_salary
Engineering,5,447000.0,89400.0
Finance,2,159000.0,79500.0
HR,2,127000.0,63500.0
Marketing,3,220000.0,73333.33333333333
Sales,3,208000.0,69333.33333333333


In [0]:
agg_df.write.mode("overwrite") \
    .saveAsTable(gold_agg_table)

In [0]:
# 2. Window Function

windowSpec = Window.partitionBy("Department").orderBy(desc("Salary"))

df_window = df.withColumn(
    "rank",
    dense_rank().over(windowSpec)
)

df_window.display()

Employee_ID,First_Name,Last_Name,Email,Department,Salary,Hire_Date,SSN,SSN_encrypted,Salary_encrypted,ingestion_date,hire_year,processed_date,full_name,salary_band,rank
EMP011,Robert,White,robert.w@example.com,Engineering,95000.0,2018-08-30,111-22-3333,2e54cc08456e5401868e64bf1ca1f597e0af65d4e111c94b514344796363fe8f,afb011cffe7baf4435f5f91deeb8f2278330b19df4fffc0a71689f3d16027883,2026-05-01,2018,2026-05-01,Robert White,High,1
EMP005,David,Wilson,david.wilson@example.com,Engineering,92000.0,2019-05-18,567-89-0123,21d6df2de747e93f696a81447ff6cd90a9c3bd2ad77ab7e5e1e7bc55c6ebc9bb,7028699272ac54450274c7ac2bf611af55953b7c3defe955bef077c6bcf74e76,2026-05-01,2019,2026-05-01,David Wilson,High,2
EMP007,James,Taylor,james.t@example.com,Engineering,88000.0,2021-02-28,789-01-2345,1ecdb8649cfa0aa0b2607ebcfb282360312884153030d41b0605a20b55182845,19d47b70a2e540349d353062a310df4163daff2971edf9d0cc0b16eb5d3ea87c,2026-05-01,2021,2026-05-01,James Taylor,High,3
EMP015,Matthew,Garcia,matthew.g@example.com,Engineering,87000.0,2020-12-01,555-66-7777,8209e227ed84ab7474b629e7a3aa59bb70f41ac76d26ab0c6496af6820d215df,5292442fad193fd38dd4ca8d52645cc4e91fa3c8ad6ca715090f2bf4e1d94613,2026-05-01,2020,2026-05-01,Matthew Garcia,High,4
EMP001,John,Doe,john.doe@example.com,Engineering,85000.0,2022-01-15,123-45-6789,01a54629efb952287e554eb23ef69c52097a75aecc0e3a93ca0855ab6d7a31a0,c08e7a2f63f203781ccfc863b16731b32563ae9b3186cd6062c518e64addf523,2026-05-01,2022,2026-05-01,John Doe,High,5
EMP012,Megan,Harris,megan.h@example.com,Finance,81000.0,2021-06-17,222-33-4444,3371a2c893de6cad595c43039412ea9d602c6e265583ba5bb9c931047ef3fb40,541574b6248c6085a282d472f720898f3c8bf3604a0b9ef1cbaa4a1a1c9807f4,2026-05-01,2021,2026-05-01,Megan Harris,High,1
EMP006,Sarah,Brown,sarah.b@example.com,Finance,78000.0,2022-09-05,678-90-1234,ba4ae52efcaa4eccd489109cb6a7c231a41f045c504512d26fb169058746c68c,a2a239b2e15e1b283bda38ca3d4fd554c76fb10227bdbf203298a31d28772e42,2026-05-01,2022,2026-05-01,Sarah Brown,Medium,2
EMP004,Emily,Davis,emily.davis@example.com,HR,65000.0,2020-07-22,456-78-9012,34450d3629c8ff56fcb8bf40ced5f8395f73a1486d9a9796afa66c4cecabc2f3,927727bfbca513547a97694ee63ce24d9b5db68631577e9971cafe555634bac7,2026-05-01,2020,2026-05-01,Emily Davis,Low,1
EMP010,Ashley,Jackson,ashley.j@example.com,HR,62000.0,2022-04-14,012-34-5678,af7d09747c8cbe325850a962be1af48ae6ddf23c98476caaa137a29be46562f0,20dd120a7502f16300940eda4f4d4fbdcf860ae54f214921e5133551d61188d5,2026-05-01,2022,2026-05-01,Ashley Jackson,Low,2
EMP008,Jessica,Anderson,jessica.a@example.com,Marketing,75000.0,2023-01-12,890-12-3456,5144e522dddd37d151df5328b9cb42c6addfab30bc1e42501c6e153bbfb43ad0,32abe77179b0106aa718c92b3ae0d64f276ca768732735495276c74fa37b8957,2026-05-01,2023,2026-05-01,Jessica Anderson,Medium,1


In [0]:
final_df = df_window.alias("a").join(
    agg_df.alias("b"),
    col("a.Department") == col("b.Department"),
    "inner"
).select(
    col("a.*"),
    col("b.total_salary"),
    col("b.avg_salary")
)

final_df.display()

Employee_ID,First_Name,Last_Name,Email,Department,Salary,Hire_Date,SSN,SSN_encrypted,Salary_encrypted,ingestion_date,hire_year,processed_date,full_name,salary_band,rank,total_salary,avg_salary
EMP011,Robert,White,robert.w@example.com,Engineering,95000.0,2018-08-30,111-22-3333,2e54cc08456e5401868e64bf1ca1f597e0af65d4e111c94b514344796363fe8f,afb011cffe7baf4435f5f91deeb8f2278330b19df4fffc0a71689f3d16027883,2026-05-01,2018,2026-05-01,Robert White,High,1,447000.0,89400.0
EMP005,David,Wilson,david.wilson@example.com,Engineering,92000.0,2019-05-18,567-89-0123,21d6df2de747e93f696a81447ff6cd90a9c3bd2ad77ab7e5e1e7bc55c6ebc9bb,7028699272ac54450274c7ac2bf611af55953b7c3defe955bef077c6bcf74e76,2026-05-01,2019,2026-05-01,David Wilson,High,2,447000.0,89400.0
EMP007,James,Taylor,james.t@example.com,Engineering,88000.0,2021-02-28,789-01-2345,1ecdb8649cfa0aa0b2607ebcfb282360312884153030d41b0605a20b55182845,19d47b70a2e540349d353062a310df4163daff2971edf9d0cc0b16eb5d3ea87c,2026-05-01,2021,2026-05-01,James Taylor,High,3,447000.0,89400.0
EMP015,Matthew,Garcia,matthew.g@example.com,Engineering,87000.0,2020-12-01,555-66-7777,8209e227ed84ab7474b629e7a3aa59bb70f41ac76d26ab0c6496af6820d215df,5292442fad193fd38dd4ca8d52645cc4e91fa3c8ad6ca715090f2bf4e1d94613,2026-05-01,2020,2026-05-01,Matthew Garcia,High,4,447000.0,89400.0
EMP001,John,Doe,john.doe@example.com,Engineering,85000.0,2022-01-15,123-45-6789,01a54629efb952287e554eb23ef69c52097a75aecc0e3a93ca0855ab6d7a31a0,c08e7a2f63f203781ccfc863b16731b32563ae9b3186cd6062c518e64addf523,2026-05-01,2022,2026-05-01,John Doe,High,5,447000.0,89400.0
EMP012,Megan,Harris,megan.h@example.com,Finance,81000.0,2021-06-17,222-33-4444,3371a2c893de6cad595c43039412ea9d602c6e265583ba5bb9c931047ef3fb40,541574b6248c6085a282d472f720898f3c8bf3604a0b9ef1cbaa4a1a1c9807f4,2026-05-01,2021,2026-05-01,Megan Harris,High,1,159000.0,79500.0
EMP006,Sarah,Brown,sarah.b@example.com,Finance,78000.0,2022-09-05,678-90-1234,ba4ae52efcaa4eccd489109cb6a7c231a41f045c504512d26fb169058746c68c,a2a239b2e15e1b283bda38ca3d4fd554c76fb10227bdbf203298a31d28772e42,2026-05-01,2022,2026-05-01,Sarah Brown,Medium,2,159000.0,79500.0
EMP004,Emily,Davis,emily.davis@example.com,HR,65000.0,2020-07-22,456-78-9012,34450d3629c8ff56fcb8bf40ced5f8395f73a1486d9a9796afa66c4cecabc2f3,927727bfbca513547a97694ee63ce24d9b5db68631577e9971cafe555634bac7,2026-05-01,2020,2026-05-01,Emily Davis,Low,1,127000.0,63500.0
EMP010,Ashley,Jackson,ashley.j@example.com,HR,62000.0,2022-04-14,012-34-5678,af7d09747c8cbe325850a962be1af48ae6ddf23c98476caaa137a29be46562f0,20dd120a7502f16300940eda4f4d4fbdcf860ae54f214921e5133551d61188d5,2026-05-01,2022,2026-05-01,Ashley Jackson,Low,2,127000.0,63500.0
EMP008,Jessica,Anderson,jessica.a@example.com,Marketing,75000.0,2023-01-12,890-12-3456,5144e522dddd37d151df5328b9cb42c6addfab30bc1e42501c6e153bbfb43ad0,32abe77179b0106aa718c92b3ae0d64f276ca768732735495276c74fa37b8957,2026-05-01,2023,2026-05-01,Jessica Anderson,Medium,1,220000.0,73333.33333333333


In [0]:
# 4. Write Gold Detail Table
final_df.write.mode("overwrite") \
    .partitionBy(config["business_partition_column"]) \
    .saveAsTable(gold_detail_table)